# AgriScan re-implementation: Kaggle "Run all"

**Before running** (right sidebar): Accelerator = **GPU T4 x2 (or P100)**, Internet = **On**, and add two inputs:
1. your private dataset with the code bundle (`agriscan_code.zip`, made by `scripts/make_kaggle_bundle.py`);
2. the public dataset **New Plant Diseases Dataset** (the code finds it by itself; no slug is hard-coded).

Then **Save Version -> Save & Run All (Commit)**, so the run continues if your browser closes.
Everything is written to `/kaggle/working/out`; see `KAGGLE_RUN.md` for what to download. Never paste tokens or passwords here.

In [ ]:
!nvidia-smi
import torch, sys
print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: turn the accelerator on (right sidebar) and restart.'

In [ ]:
import glob, os, shutil, subprocess, zipfile
WORK = '/kaggle/working/agriscan'
GITHUB_URL = None   # alternative to the bundle once the repo is public, e.g. 'https://github.com/<user>/agriscan-reimpl'

def find_code():
    for pat in ('/kaggle/input/*/pyproject.toml', '/kaggle/input/*/*/pyproject.toml'):
        for p in glob.glob(pat):
            if 'name = "agriscan"' in open(p).read():
                return os.path.dirname(p)
    for z in glob.glob('/kaggle/input/*/*.zip'):
        if 'agriscan' in os.path.basename(z):
            zipfile.ZipFile(z).extractall('/kaggle/working/_unzipped')
            return '/kaggle/working/_unzipped'
    return None

src = find_code()
if src:
    shutil.copytree(src, WORK, dirs_exist_ok=True)
    print('code copied from', src)
elif GITHUB_URL:
    subprocess.run(['git', 'clone', GITHUB_URL, WORK], check=True)
else:
    raise SystemExit('Code not found: add the private code-bundle dataset as an input (see KAGGLE_RUN.md), or set GITHUB_URL.')
print(open(WORK + '/BUNDLE_INFO.json').read() if os.path.exists(WORK + '/BUNDLE_INFO.json') else 'git clone, no BUNDLE_INFO.json')

In [ ]:
# Kaggle ships torch/torchvision; this adds the rest (ultralytics = YOLO stage, pytest = quick self-test).
!pip install -q -e "/kaggle/working/agriscan[dev,app,detect]"

In [ ]:
# 1-2 minutes on CPU: logic tests on synthetic data. A failure here means do not trust the run below.
!cd /kaggle/working/agriscan && python -m pytest -q -x tests

In [ ]:
# The whole experiment. Safe to re-run after a crash: finished stages are skipped, unfinished ones resume.
# To continue after the SESSION died: add the previous version's output as an input and set RESUME_FROM to it.
RESUME_FROM = None   # e.g. '/kaggle/input/agriscan-run1/out'
cmd = 'cd /kaggle/working/agriscan && python scripts/run_all.py --data auto --workdir /kaggle/working/out --max-hours 10'
if RESUME_FROM:
    cmd += f' --resume-from {RESUME_FROM}'
!{cmd}

In [ ]:
# What to download (Output tab): export/agriscan_results.zip and export/models/*.pt
!ls -la /kaggle/working/out/export /kaggle/working/out/export/models
!cat /kaggle/working/out/results/summary.md